# Fine-tune Mistral-7B-Instruct-v0.3 for the Bangladesh Government Chatbot

This notebook is adapted from the supplied Qwen fine-tuning notebook, but uses **Mistral-7B-Instruct-v0.3** and Mistral's own chat template.

**Training setup**
- Base: `unsloth/mistral-7b-instruct-v0.3-bnb-4bit`
- Method: 4-bit QLoRA with Unsloth
- LoRA rank: 16
- Epochs: 3
- Dataset: `Govt_Chatbot/RAG/unified_train.json` (938 examples in the supplied project)
- Task: Bengali Bangladesh-government-service question answering

> Kaggle: enable a GPU accelerator before running. A T4/P100-class GPU is appropriate for this QLoRA setup.

**Kaggle note:** enable Internet access unless this Hugging Face model is already cached/attached to the notebook.

In [ ]:
# Kaggle environment check
import os, sys, glob, json, zipfile, shutil, random
import numpy as np
import pandas as pd
import torch

print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('BF16 supported:', torch.cuda.is_bf16_supported())
else:
    print('WARNING: No GPU detected. Enable a Kaggle GPU before training.')

In [ ]:
# Install/upgrade the training stack
!pip install -q -U unsloth unsloth_zoo transformers trl datasets accelerate peft bitsandbytes

In [ ]:
# IMPORTANT: if Kaggle asks for a runtime restart after installation, restart once
# and resume from THIS cell. This cell is self-contained.
import os, sys, glob, json, zipfile, shutil, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch

from unsloth import FastLanguageModel, is_bf16_supported
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

SEED = 3407
MAX_SEQ_LENGTH = 1024
BASE_MODEL = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"
OUTPUT_DIR = "/kaggle/working/government_mistral_lora"
TRAINER_DIR = "/kaggle/working/mistral-government-checkpoints"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Base model:", BASE_MODEL)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## Locate the training data

The supplied project stores the training set at `Govt_Chatbot/RAG/unified_train.json`. The cell below is intentionally flexible: it first looks for that JSON directly in Kaggle inputs, then for the older `government_chat_train.jsonl`, and finally inside any attached ZIP.

In [ ]:
from pathlib import Path


def locate_training_file():
    # 1) Preferred supplied-project file
    candidates = glob.glob('/kaggle/input/**/unified_train.json', recursive=True)
    if candidates:
        return candidates[0]

    # 2) Backward compatibility with the original Qwen notebook
    candidates = glob.glob('/kaggle/input/**/government_chat_train.jsonl', recursive=True)
    if candidates:
        return candidates[0]

    # 3) Look inside attached ZIP files
    for zip_path in glob.glob('/kaggle/input/**/*.zip', recursive=True):
        try:
            with zipfile.ZipFile(zip_path) as zf:
                members = zf.namelist()
                preferred = [m for m in members if m.endswith('Govt_Chatbot/RAG/unified_train.json')]
                fallback = [m for m in members if m.endswith('unified_train.json')]
                matches = preferred or fallback
                if matches:
                    extract_dir = Path('/kaggle/working/mistral_train_data')
                    extract_dir.mkdir(parents=True, exist_ok=True)
                    member = matches[0]
                    zf.extract(member, extract_dir)
                    return str(extract_dir / member)
        except zipfile.BadZipFile:
            pass

    raise FileNotFoundError(
        'Could not find unified_train.json or government_chat_train.jsonl. '
        'Attach the Govt_Chatbot ZIP/dataset to the Kaggle notebook.'
    )

TRAIN_DATA_PATH = locate_training_file()
print('Training file:', TRAIN_DATA_PATH)

In [ ]:
# Load raw training examples
raw_dataset = load_dataset(
    "json",
    data_files=TRAIN_DATA_PATH,
    split="train",
)

print(raw_dataset)
print("Columns:", raw_dataset.column_names)
print("Examples:", len(raw_dataset))
print("\nFirst example:")
print(raw_dataset[0])

required = {"instruction", "input", "output"}
missing = required - set(raw_dataset.column_names)
assert not missing, f"Missing required columns: {missing}"


In [ ]:
# Dataset diagnostics
pdf = raw_dataset.to_pandas()

for col in ["instruction", "input", "output"]:
    pdf[col] = pdf[col].fillna("").astype(str)

print("Examples:", len(pdf))
print("Non-empty input fields:", (pdf["input"].str.strip() != "").sum())
print("Average instruction characters:", round(pdf["instruction"].str.len().mean(), 1))
print("Average answer characters:", round(pdf["output"].str.len().mean(), 1))
print("Maximum answer characters:", int(pdf["output"].str.len().max()))

if "domain" in pdf.columns:
    print("\nDomain counts:")
    print(pdf["domain"].value_counts())


## Load Mistral-7B in 4-bit and attach LoRA

The seven projection modules below match the LoRA setup used in your Qwen experiment and are also the standard Unsloth recommendation for strong QLoRA adaptation.

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('EOS token:', repr(tokenizer.eos_token))
print('PAD token:', repr(tokenizer.pad_token))
print('Tokenizer vocab size:', len(tokenizer))

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        'q_proj',
        'k_proj',
        'v_proj',
        'o_proj',
        'gate_proj',
        'up_proj',
        'down_proj',
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
)

model.print_trainable_parameters()

## Format the 938 examples with Mistral's tokenizer

Do **not** use Qwen's `<|im_start|> ... <|im_end|>` tokens here. `tokenizer.apply_chat_template(...)` lets the installed Mistral tokenizer produce its native instruction format.

In [ ]:
SYSTEM_PROMPT = (
    'তুমি বাংলাদেশ সরকারের সরকারি সেবা সম্পর্কিত একজন সহায়ক সহকারী। '
    'প্রশ্নের সরাসরি, সংক্ষিপ্ত এবং নির্ভুল উত্তর বাংলায় দাও। '
    'শুধুমাত্র নির্ভরযোগ্য তথ্য দাও। '
    "যদি তথ্য জানা না থাকে, বলবে 'আমি জানি না'।"
)


def formatting(example):
    instruction = str(example.get('instruction', '') or '').strip()
    input_text = str(example.get('input', '') or '').strip()
    answer = str(example.get('output', '') or '').strip()

    user_text = instruction
    if input_text:
        user_text += '\n' + input_text

    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': user_text},
        {'role': 'assistant', 'content': answer},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    # Most Mistral templates already terminate assistant turns with EOS.
    # Add it only if the rendered template did not do so.
    if tokenizer.eos_token and not text.endswith(tokenizer.eos_token):
        text += tokenizer.eos_token

    return {'text': text}

formatted_dataset = raw_dataset.map(
    formatting,
    remove_columns=raw_dataset.column_names,
)

print(formatted_dataset)

In [ ]:
# Inspect one rendered training sample and its token length
sample_text = formatted_dataset[0]["text"]
print(sample_text)
print("\nToken length:", len(tokenizer(sample_text, add_special_tokens=False)["input_ids"]))


In [ ]:
# Check truncation risk across all 938 examples before training
lengths = []
for row in formatted_dataset:
    lengths.append(len(tokenizer(row['text'], add_special_tokens=False)['input_ids']))

lengths = np.array(lengths)
print('Token length median:', int(np.median(lengths)))
print('Token length p95:', int(np.percentile(lengths, 95)))
print('Token length max:', int(lengths.max()))
print(f'Examples longer than {MAX_SEQ_LENGTH}:', int((lengths > MAX_SEQ_LENGTH).sum()))

if (lengths > MAX_SEQ_LENGTH).any():
    print('WARNING: Increase MAX_SEQ_LENGTH before training if this count is not zero.')

## Train

This keeps the main Qwen experiment hyperparameters so the comparison is meaningful: rank 16, alpha 16, batch 2, accumulation 4, learning rate `2e-4`, and 3 epochs. Unsloth recommends roughly 1–3 epochs for instruction datasets as a good starting range.

In [ ]:
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=formatted_dataset,
    args=SFTConfig(
        output_dir=TRAINER_DIR,

        # Dataset handling
        dataset_text_field='text',
        max_length=MAX_SEQ_LENGTH,
        packing=True,
        dataset_num_proc=2,

        # Training
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,

        # Optimizer
        learning_rate=2e-4,
        optim='adamw_8bit',
        lr_scheduler_type='cosine',
        warmup_ratio=0.03,
        weight_decay=0.01,

        # Precision
        fp16=not is_bf16_supported(),
        bf16=is_bf16_supported(),

        # Stability / logging
        max_grad_norm=1.0,
        logging_steps=10,
        save_strategy='epoch',
        save_total_limit=2,
        report_to='none',
        seed=SEED,
    ),
)

print('Trainer ready.')

In [ ]:
# Start the full fine-tuning run
trainer_stats = trainer.train()
trainer_stats

In [ ]:
# Save LoRA adapters + tokenizer
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Save training history for later comparison
history_path = '/kaggle/working/mistral_training_history.csv'
pd.DataFrame(trainer.state.log_history).to_csv(history_path, index=False)

run_config = {
    'stage': 'Mistral-7B QLoRA fine-tuning',
    'base_model': BASE_MODEL,
    'training_examples': len(raw_dataset),
    'max_seq_length': MAX_SEQ_LENGTH,
    'lora_r': 16,
    'lora_alpha': 16,
    'lora_dropout': 0.0,
    'target_modules': ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    'num_train_epochs': 3,
    'per_device_train_batch_size': 2,
    'gradient_accumulation_steps': 4,
    'learning_rate': 2e-4,
    'seed': SEED,
}
with open('/kaggle/working/mistral_finetune_run_config.json', 'w', encoding='utf-8') as f:
    json.dump(run_config, f, ensure_ascii=False, indent=2)

print('Saved adapter:', OUTPUT_DIR)
print('Saved history:', history_path)

In [ ]:
# Create a downloadable ZIP of the LoRA adapter
zip_path = shutil.make_archive(
    '/kaggle/working/government_mistral_lora',
    'zip',
    OUTPUT_DIR,
)
print('Created:', zip_path)

## Sanity-check the fine-tuned model

The same tokenizer/chat template is used at inference time. This is essential; using a different template after training can cause gibberish, repetition, or poor stopping behavior.

In [ ]:
FastLanguageModel.for_inference(model)


def generate_answer(question, context=''):
    question = str(question or '').strip()
    context = '' if context is None else str(context).strip()

    if context:
        user_text = (
            f'প্রশ্ন: {question}\n\n'
            'প্রাসঙ্গিক তথ্য:\n'
            f'{context}\n\n'
            'উপরের প্রাসঙ্গিক তথ্য ব্যবহার করে প্রশ্নের সরাসরি ও সংক্ষিপ্ত উত্তর বাংলায় দাও।'
        )
    else:
        user_text = question

    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': user_text},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            use_cache=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            repetition_penalty=1.05,
        )

    generated = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

print(generate_answer('NID আবেদন করতে কোথায় যেতে হবে?'))

## Optional: test a RAG-style prompt

Use the exact same `generate_answer(question, context)` function in your Mistral RAG notebook. Start with the **top 3 reranked passages**, not all five, then compare Top-1/Top-3/Top-5 experimentally.

In [ ]:
# Example only — replace the context with text from your retriever.
example_context = (
    'ই-পাসপোর্ট আবেদনের সাধারণ পাঁচটি ধাপ হলো: প্রথমে আবেদনকারীর এলাকায় '
    'ই-পাসপোর্ট সেবা চালু আছে কি না যাচাই করা, এরপর অনলাইনে আবেদন পূরণ করা, '
    'নির্ধারিত ফি পরিশোধ করা, পাসপোর্ট অফিসে গিয়ে বায়োমেট্রিক এনরোলমেন্ট '
    'সম্পন্ন করা এবং শেষে পাসপোর্ট সংগ্রহ করা।'
)

print(generate_answer('ই-পাসপোর্ট আবেদনের পাঁচটি মূল ধাপ কী কী?', example_context))

## Outputs to keep from Kaggle

After the notebook finishes, save these files from `/kaggle/working/`:

- `government_mistral_lora.zip` — the trained LoRA adapter
- `government_mistral_lora/` — unzipped adapter + tokenizer
- `mistral_training_history.csv` — loss/log history
- `mistral_finetune_run_config.json` — reproducibility settings

You do **not** need to merge the full 7B model just to use it in your RAG pipeline. Load the same Mistral base model together with this adapter.